In [ ]:
# this code must be run after training or reloading the GCN model
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch_geometric.data import Data

# Pick any test graph (total test graph are 84) that we want to visualize
g = test_graphs[3]
data = Data(
    x=torch.tensor(g["X_flat"], dtype=torch.float),
    edge_index=edge_index,
    num_nodes=g["X_flat"].shape[0]
).to(device)

with torch.no_grad():
    embeddings, _ = model(data.x, data.edge_index)

embeddings = embeddings.cpu().numpy()   

ny, nx = 450, 224
n_channels = embeddings.shape[1]

embeddings_scaled = np.zeros_like(embeddings)
for ch in range(n_channels):
    channel_data = embeddings[:, ch]
    vmin = np.percentile(channel_data, 2)
    vmax = np.percentile(channel_data, 98)
    if vmax > vmin:
        embeddings_scaled[:, ch] = np.clip((channel_data - vmin) / (vmax - vmin), 0, 1)
    else:
        embeddings_scaled[:, ch] = channel_data

fig, axes = plt.subplots(2, 4, figsize=(12, 18), dpi=300)
axes = axes.flatten()

for ch in range(n_channels):
    ax = axes[ch]
    emb_grid = embeddings_scaled[:, ch].reshape(ny, nx)

    im = ax.imshow(
        emb_grid,
        origin='lower',
        cmap='viridis',
        aspect=ny / nx,
        interpolation='bilinear'
    )
    ax.set_title(f'Channel {ch}', fontsize=10)
    ax.set_xlabel('x-index')
    ax.set_ylabel('y-index')
    ax.tick_params(labelsize=8)

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Normalized GCN embedding magnitude', fontsize=10)
plt.subplots_adjust(left=0.05, right=0.9, top=0.95, bottom=0.05, wspace=0.3, hspace=0.3)
plt.show()

# If you want to visualize a specific channel in more detail, you can do that separately as well:
# fig2, ax2 = plt.subplots(1, 1, figsize=(4, 4), dpi=300)  
# ch = 3  # Selected channel index (0–7); the 3rd convolutional layer has 8 channels
# emb_grid = embeddings_scaled[:, ch].reshape(ny, nx)
# im2 = ax2.imshow(
#     emb_grid,
#     origin='lower',
#     cmap='viridis',
#     aspect=ny / nx,
#     interpolation='bilinear'
# )

# ax2.set_xlabel('x-index', fontsize=8)
# ax2.set_ylabel('y-index', fontsize=8)
# ax2.tick_params(labelsize=6)
# cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8, ticks=np.linspace(0, 1, 5))
# cbar2.set_label('GCN embedding magnitude', fontsize=8)
# cbar2.ax.tick_params(labelsize=8)  # Yahan numbers ka size set karo

# plt.tight_layout()
# plt.show()